# Lesson 4: Generalized Advantage Estimation / GAE on CartPole

GAE improves how we calculate advantages.

A2C used:

```text
advantage = return - value
```

GAE uses temporal-difference errors over multiple future steps:

```text
delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)
```

Then it smooths those deltas with lambda.

The goal is a better bias/variance tradeoff.


## 1) Imports


In [ ]:
%pip install -U "gymnasium[classic-control]"

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)


## 2) Seeds and Device


In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3) Environments


In [ ]:
train_env = gym.make("CartPole-v1")
test_env = gym.make("CartPole-v1")

train_env.action_space.seed(SEED)
test_env.action_space.seed(SEED + 1)

state, info = train_env.reset(seed=SEED)
print("example state:", state)
print("observation space:", train_env.observation_space)
print("action space:", train_env.action_space)


## 4) Actor-Critic Network


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.actor = nn.Linear(hidden_dim, output_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return logits, value


## 5) Build Policy and Optimizer


In [ ]:
INPUT_DIM = train_env.observation_space.shape[0]
HIDDEN_DIM = 128
OUTPUT_DIM = train_env.action_space.n

policy = ActorCritic(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)

print(policy)


## 6) Select an Action


In [ ]:
def select_action(policy, state):
    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    logits, value = policy(state_tensor)
    distribution = Categorical(logits=logits)
    action = distribution.sample()
    log_prob = distribution.log_prob(action)
    entropy = distribution.entropy()
    return action.item(), log_prob.squeeze(0), value.squeeze(0), entropy.squeeze(0)


## 7) GAE Calculation

GAE works backwards through an episode.

At each step:

```text
delta = reward + gamma * next_value * mask - value
advantage = delta + gamma * lambda * mask * future_advantage
```

`mask` is 0 when the episode ended, so we do not bootstrap beyond terminal states.


In [ ]:
def calculate_gae(rewards, values, next_values, dones, discount_factor, gae_lambda, normalize=True):
    values = torch.stack(values)
    next_values = torch.stack(next_values)
    dones = torch.as_tensor(dones, dtype=torch.float32, device=device)
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)

    advantages = torch.zeros_like(rewards, device=device)
    gae = 0.0

    for t in reversed(range(len(rewards))):
        mask = 1.0 - dones[t]
        delta = rewards[t] + discount_factor * next_values[t] * mask - values[t]
        gae = delta + discount_factor * gae_lambda * mask * gae
        advantages[t] = gae

    returns = advantages + values

    if normalize and len(advantages) > 1:
        std = advantages.std(unbiased=False)
        if std > 1e-8:
            advantages = (advantages - advantages.mean()) / (std + 1e-8)

    return advantages, returns, values


## 8) Update Policy

The actor uses GAE advantages. The critic learns the GAE-based returns.


In [ ]:
def update_policy(advantages, log_probs, returns, values, entropies, optimizer):
    log_probs = torch.stack(log_probs)
    entropies = torch.stack(entropies)

    policy_loss = -(advantages.detach() * log_probs).sum()
    value_loss = F.mse_loss(values, returns.detach())
    entropy_bonus = entropies.mean()

    loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return policy_loss.item(), value_loss.item(), loss.item()


## 9) Train One Episode With GAE

We need `value` and `next_value` at every step so we can compute TD errors.


In [ ]:
def train_one_episode(env, policy, optimizer, discount_factor, gae_lambda, seed=None):
    policy.train()

    log_probs = []
    values = []
    next_values = []
    rewards = []
    dones = []
    entropies = []
    episode_reward = 0.0

    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action, log_prob, value, entropy = select_action(policy, state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        with torch.no_grad():
            next_state_tensor = torch.as_tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)
            _, next_value = policy(next_state_tensor)
            next_value = next_value.squeeze(0)

        log_probs.append(log_prob)
        values.append(value)
        next_values.append(next_value)
        rewards.append(reward)
        dones.append(float(done))
        entropies.append(entropy)
        episode_reward += reward
        state = next_state

    advantages, returns, values = calculate_gae(
        rewards, values, next_values, dones, discount_factor, gae_lambda
    )
    policy_loss, value_loss, total_loss = update_policy(
        advantages, log_probs, returns, values, entropies, optimizer
    )

    return policy_loss, value_loss, total_loss, episode_reward


## 10) Evaluate


In [ ]:
def evaluate(env, policy, seed=None):
    policy.eval()

    episode_reward = 0.0
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()

        state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

    return episode_reward


## 11) Training Loop


In [ ]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
GAE_LAMBDA = 0.95
N_TRIALS = 25
REWARD_THRESHOLD = 475
PRINT_EVERY = 10

train_rewards = []
test_rewards = []
policy_losses = []
value_losses = []
recent_test_rewards = deque(maxlen=N_TRIALS)

for episode in range(1, MAX_EPISODES + 1):
    policy_loss, value_loss, total_loss, train_reward = train_one_episode(
        train_env, policy, optimizer, DISCOUNT_FACTOR, GAE_LAMBDA, seed=SEED + episode
    )
    test_reward = evaluate(test_env, policy, seed=SEED + 10_000 + episode)

    train_rewards.append(train_reward)
    test_rewards.append(test_reward)
    policy_losses.append(policy_loss)
    value_losses.append(value_loss)
    recent_test_rewards.append(test_reward)

    if episode % PRINT_EVERY == 0:
        print(
            f"| Episode: {episode:3} | "
            f"Mean Train: {np.mean(train_rewards[-N_TRIALS:]):6.1f} | "
            f"Mean Test: {np.mean(recent_test_rewards):6.1f} | "
            f"Policy Loss: {policy_loss:8.2f} | Value Loss: {value_loss:8.2f} |"
        )

    if len(recent_test_rewards) == N_TRIALS and np.mean(recent_test_rewards) >= REWARD_THRESHOLD:
        print(f"Reached reward threshold in {episode} episodes")
        break


## 12) Plot Rewards


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train_rewards, label="Train Reward", alpha=0.7)
plt.plot(test_rewards, label="Test Reward")
plt.axhline(REWARD_THRESHOLD, color="red", linestyle="--", label="Threshold")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()


## 13) Watch Policy


In [ ]:
def watch_policy(policy, seed=SEED, max_steps=500):
    render_env = gym.make("CartPole-v1", render_mode="human")
    state, info = render_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated) and steps < max_steps:
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()
        state, reward, terminated, truncated, info = render_env.step(action)
        total_reward += reward
        steps += 1

    print(f"Episode finished. Total reward: {total_reward}, steps: {steps}")
    render_env.close()


# Uncomment after training if your machine supports GUI rendering.
# watch_policy(policy)


## 14) Exercises

1. Change `GAE_LAMBDA` to `0.0`, `0.5`, and `1.0`. What changes?
2. Print `delta` values inside `calculate_gae` for one short episode.
3. Explain why GAE is a compromise between one-step TD and Monte Carlo returns.

## 15) Mental Model

GAE asks:

```text
Across several future steps, was the outcome better than the critic predicted?
```

`lambda` controls how much future TD error gets included.
